# Retail Orders — KPI Data Quality Profile

Executable checks for completeness, uniqueness, validity, consistency and freshness.

**Expected grain:** one row per unique `order_id`  
**Refresh cadence:** daily

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = 'data/retail-orders-raw.csv'
REPORTING_DATE = pd.Timestamp.today().normalize().date()

df = pd.read_csv(DATA_PATH)

print('Rows:', len(df))
print('Columns:', len(df.columns))
display(df.head())

In [ ]:
print('--- DATA TYPES ---')
display(df.dtypes.to_frame('dtype'))

print('--- MISSING VALUES ---')
display(df.isna().sum().to_frame('missing_count'))

print('--- DUPLICATE ROWS ---')
print(df.duplicated().sum())

print('--- DUPLICATE ORDER IDs ---')
print(df['order_id'].duplicated(keep=False).sum())

In [ ]:
cat_maps = {
    'customer_segment': {
        'student':'Student', 'fresher':'Fresher', 'professional':'Professional'
    },
    'category': {
        'learning kit':'Learning Kit', 'course access':'Course Access', 'mentor session':'Mentor Session'
    },
    'payment_status': {
        'paid':'Paid', 'pending':'Pending', 'failed':'Failed', 'refunded':'Refunded'
    }
}

for col, mapping in cat_maps.items():
    df[col + '_normalized'] = (
        df[col].astype('string').str.strip().str.lower().map(mapping)
    )

df['order_date_parsed'] = pd.to_datetime(df['order_date'], errors='coerce')
df['quantity_num'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price_num'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['discount_pct_num'] = pd.to_numeric(df['discount_pct'], errors='coerce')

display(df.head())

In [ ]:
def pct(mask):
    return round(mask.mean() * 100, 2)

order_id_complete = df['order_id'].notna()
order_id_unique = order_id_complete & ~df['order_id'].duplicated(keep=False)
date_complete = df['order_date'].notna()
city_complete = df['city'].notna() & df['city'].astype('string').str.strip().ne('')

checks = [
    ['Completeness','order_id',pct(order_id_complete),100,order_id_complete.all()],
    ['Uniqueness','order_id',pct(order_id_unique),100,order_id_unique.all()],
    ['Completeness','order_date',pct(date_complete),100,date_complete.all()],
    ['Completeness','city',pct(city_complete),98,pct(city_complete) >= 98],
]

quality_summary = pd.DataFrame(checks, columns=['Dimension','Field','Observed %','Threshold %','Pass'])
display(quality_summary)

In [ ]:
date_valid = df['order_date_parsed'].notna()
quantity_valid = df['quantity_num'].notna() & (df['quantity_num'] > 0) & (df['quantity_num'] % 1 == 0)
price_valid = df['unit_price_num'].notna() & (df['unit_price_num'] >= 0)
discount_valid = df['discount_pct_num'].notna() & df['discount_pct_num'].between(0, 100)
customer_valid = df['customer_segment_normalized'].notna()
category_valid = df['category_normalized'].notna()
payment_valid = df['payment_status_normalized'].notna()

validity_summary = pd.DataFrame([
    ['Validity','order_date',pct(date_valid),100,date_valid.all()],
    ['Validity','quantity',pct(quantity_valid),100,quantity_valid.all()],
    ['Validity','unit_price',pct(price_valid),100,price_valid.all()],
    ['Validity','discount_pct',pct(discount_valid),100,discount_valid.all()],
    ['Validity','customer_segment',pct(customer_valid),100,customer_valid.all()],
    ['Validity','category',pct(category_valid),100,category_valid.all()],
    ['Consistency','payment_status',pct(payment_valid),100,payment_valid.all()],
], columns=['Dimension','Field','Observed %','Threshold %','Pass'])

display(validity_summary)

In [ ]:
valid_dates = df.loc[df['order_date_parsed'].notna(), 'order_date_parsed']

if len(valid_dates):
    latest_date = valid_dates.max().date()
    freshness_lag = (REPORTING_DATE - latest_date).days
else:
    latest_date = None
    freshness_lag = None

freshness_pass = freshness_lag is not None and freshness_lag <= 1

display(pd.DataFrame([{
    'Reporting Date': REPORTING_DATE,
    'Latest Valid Order Date': latest_date,
    'Lag Days': freshness_lag,
    'Threshold Days': 1,
    'Pass': freshness_pass
}]))

In [ ]:
failure_mask = (
    df['order_id'].isna()
    | df['order_id'].duplicated(keep=False)
    | df['order_date_parsed'].isna()
    | ~quantity_valid
    | ~price_valid
    | ~discount_valid
    | ~customer_valid
    | ~category_valid
    | ~payment_valid
)

failed_rows = df.loc[failure_mask].copy()
print('Rows requiring correction/quarantine:', len(failed_rows))

display(failed_rows)


In [ ]:
critical_pass = all([
    order_id_complete.all(),
    order_id_unique.all(),
    date_complete.all(),
    date_valid.all(),
    quantity_valid.all(),
    price_valid.all(),
    discount_valid.all(),
    customer_valid.all(),
    category_valid.all(),
    payment_valid.all(),
])

contract_pass = bool(critical_pass and freshness_pass and (pct(city_complete) >= 98))

print('=' * 55)
print('DATA QUALITY CONTRACT RESULT')
print('=' * 55)
print('PASS' if contract_pass else 'FAIL')

if not contract_pass:
    print('Action: BLOCK KPI PUBLICATION')
    print('Correct/quarantine failed records, refresh the data, and rerun the notebook.')

## Interpretation

A FAIL result is expected when the supplied raw dataset violates any critical contract rule. The failure itself is evidence that the quality checks are executable and capable of preventing unreliable KPI publication.